# Clasificación Iris con XGBoost y MLflow 3

Flujo Databricks completo usando APIs oficiales: scikit-learn para el split, MLflow para tracking, tracing, evaluación, tablas y registro en Unity Catalog.

## 1. Dependencias

La librería Databricks se instala antes de los imports. Ejecuta esta celda en un notebook Databricks y reinicia Python después de `%pip`.

In [0]:
%pip install ./tools "mlflow[databricks]>=3.1,<4"
dbutils.library.restartPython()

## 2. Configuración e imports

`build_config` y `load_dataset` solo gestionan configuración y validación. El tracking permanece explícito en este notebook mediante las APIs oficiales.

In [0]:
import json
import tempfile

import matplotlib.pyplot as plt
import mlflow
import mlflow.pyfunc
import mlflow.xgboost
from mlflow.models import evaluate, infer_signature
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

from iris_mlflow_utils import (
    build_classification_table,
    build_config,
    build_metrics_summary_table,
    ensure_feature_table,
    evaluate_train_test,
    load_dataset_frame,
)

config = build_config(
    model_slug="xgboost",
    registered_model_name="workspace.default.iris_classifier",
    run_name="xgboost-iris-challenger",
)
model_params = {
    "n_estimators": 100,
    "max_depth": 4,
    "learning_rate": 0.10,
    "subsample": 0.90,
    "colsample_bytree": 0.90,
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "random_state": config.random_state,
    "n_jobs": 2,
}
if config.tracking_uri:
    mlflow.set_tracking_uri(config.tracking_uri)
mlflow.set_registry_uri(config.registry_uri)
mlflow.set_experiment(config.experiment_name)
print(f"Dataset: {config.dataset_path}")
print(f"Experimento: {config.experiment_name}")
FEATURE_TABLE = "workspace.default.iris_features"
print(f"Modelo: {config.registered_model_name}")
print(f"Feature table: {FEATURE_TABLE}")

## 3. Carga y split oficial

Los spans reciben solamente resúmenes. `train_test_split` es la implementación oficial de scikit-learn y conserva la estratificación de clases.

In [0]:
with mlflow.start_span(name="iris.load_dataset", span_type="CHAIN", attributes={"dataset_version": config.dataset_version}) as span:
    feature_table_created = ensure_feature_table(
        spark, table_name=FEATURE_TABLE, dataset_path=config.dataset_path
    )
    feature_spark_df = spark.table(FEATURE_TABLE)
    dataset = load_dataset_frame(feature_spark_df.toPandas())
    feature_table_exists = not feature_table_created
    span.set_outputs({"rows": len(dataset.dataframe), "features": len(dataset.feature_columns), "feature_table_created": feature_table_created})

with mlflow.start_span(name="iris.split_dataset", span_type="CHAIN", attributes={"random_state": config.random_state, "test_size": config.test_size}) as span:
    x_train, x_test, y_train, y_test = train_test_split(
        dataset.features, dataset.target, test_size=config.test_size, random_state=config.random_state, stratify=dataset.target
    )
    span.set_outputs({"train_rows": len(x_train), "test_rows": len(x_test)})
print(f"Train: {len(x_train)} | Test: {len(x_test)} | Clases: {list(dataset.classes)}")
display(dataset.dataframe.head())

## 4. Run, entrenamiento, evaluación y registro oficial

Todo el ciclo usa directamente `mlflow.start_run`, `mlflow.start_span`, `mlflow.models.evaluate`, `mlflow.log_table` y `mlflow.xgboost.log_model`. La URI se toma de `model_info.model_uri`; nunca se reconstruye como `runs:/...`.

In [0]:
model = XGBClassifier(num_class=len(dataset.classes), **model_params)
with mlflow.start_run(run_name=config.run_name) as run:
    run_id = run.info.run_id
    mlflow.log_params({**model_params, "test_size": config.test_size, "random_state": config.random_state, "dataset_version": config.dataset_version, "feature_columns": json.dumps(dataset.feature_columns), "feature_table": FEATURE_TABLE, "feature_table_exists": str(feature_table_exists).lower(), "feature_table_created": str(feature_table_created).lower()})
    mlflow.set_tags({"model_type": "XGBoost", "dataset_version": config.dataset_version, "project_version": config.project_version, "primary_metric": config.primary_metric, "tracking_backend": "databricks-managed" if not config.tracking_uri else config.tracking_uri, "feature_table_source": "unity_catalog", "evaluation_status": "started"})
    with mlflow.start_span(name="iris.train.XGBoost", span_type="CHAIN", attributes={"model_type": "XGBoost", "train_rows": len(x_train)}) as span:
        model.fit(x_train, y_train)
        span.set_outputs({"status": "completed"})
    evaluations = evaluate_train_test(model, x_train, x_test, y_train, y_test, len(dataset.classes))
    metrics = {f"{partition}_{name}": value for partition, result in evaluations.items() for name, value in result.metrics.items()}
    mlflow.log_metrics(metrics)
    mlflow.log_dict(evaluations["test"].report, "evaluation/classification_report.json")
    mlflow.log_dict({str(i): name for i, name in enumerate(dataset.classes)}, "class_mapping.json")
    metrics_summary = build_metrics_summary_table(evaluations, model_type="XGBoost", dataset_version=config.dataset_version, project_version=config.project_version, run_id=run_id)
    classification_by_class = build_classification_table(evaluations, dataset.classes, model_type="XGBoost", dataset_version=config.dataset_version, project_version=config.project_version, run_id=run_id)
    mlflow.log_table(metrics_summary, "evaluation/metrics_summary.json")
    mlflow.log_table(classification_by_class, "evaluation/classification_by_class.json")
    figure, axis = plt.subplots(figsize=(6, 5))
    axis.imshow(evaluations["test"].confusion_matrix, cmap="Blues")
    axis.set(xticks=range(len(dataset.classes)), yticks=range(len(dataset.classes)), xticklabels=dataset.classes, yticklabels=dataset.classes, xlabel="Predicción", ylabel="Valor real", title="Matriz de confusión")
    figure.tight_layout()
    with tempfile.TemporaryDirectory() as directory:
        path = f"{directory}/confusion_matrix.png"
        figure.savefig(path, dpi=150)
        mlflow.log_artifact(path, artifact_path="evaluation")
    plt.close(figure)
    signature = infer_signature(x_train, model.predict(x_train))
    with mlflow.start_span(name="iris.log_model", span_type="CHAIN", attributes={"model_type": "XGBoost"}) as span:
        model_info = mlflow.xgboost.log_model(model, name="model", signature=signature, input_example=x_train.head(5), registered_model_name=config.registered_model_name)
        span.set_outputs({"model_uri": model_info.model_uri, "model_id": model_info.model_id})
    evaluation_data = x_test.copy()
    evaluation_data["target"] = y_test
    with mlflow.start_span(name="iris.mlflow_evaluate", span_type="CHAIN", attributes={"model_type": "XGBoost", "test_rows": len(x_test)}) as span:
        evaluation_result = evaluate(model=model_info.model_uri, model_id=model_info.model_id, data=evaluation_data, targets="target", model_type="classifier", evaluator_config={"log_model_explainability": False})
        span.set_outputs({"metrics": evaluation_result.metrics})
    mlflow.log_metrics({f"mlflow_eval_{name}": float(value) for name, value in evaluation_result.metrics.items() if isinstance(value, (int, float))})
    mlflow.set_tag("evaluation_status", "completed")

## 5. Validación del run y carga del modelo

`MlflowClient` verifica que el run tenga parámetros, métricas y artefactos. El modelo se carga con la URI oficial retornada por `log_model`.

In [0]:
client = mlflow.MlflowClient()
logged_run = client.get_run(run_id)
if not logged_run.data.params or not logged_run.data.metrics:
    raise RuntimeError(f"El run {run_id} no contiene tracking completo.")
artifact_paths = {artifact.path for artifact in client.list_artifacts(run_id, "evaluation")}
required = {"evaluation/metrics_summary.json", "evaluation/classification_by_class.json", "evaluation/classification_report.json", "evaluation/confusion_matrix.png"}
if not required.issubset(artifact_paths):
    raise RuntimeError(f"Faltan artefactos de evaluación en {run_id}: {required - artifact_paths}")
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)
print(f"Experiment ID: {run.info.experiment_id}")
print(f"Run ID: {run_id}")
print(f"Model ID: {model_info.model_id}")
print(f"Model URI: {model_info.model_uri}")
print(f"Registered version: {getattr(model_info, 'registered_model_version', None)}")
print(f"Trace ID: {mlflow.get_last_active_trace_id()}")
print(f"Evaluation metrics: {evaluation_result.metrics}")
print(f"Sample predictions: {loaded_model.predict(x_test.head(3)).tolist()}")